In [ ]:
#1. Scrap data  RUN CELL 1 AND CELL 4 AND ONWARDS
import requests
from bs4 import BeautifulSoup
import pandas as pd


urls = [
    "https://www.uia.no/english/studies/courses/2026/spring/ikt469.html",
    "https://www.uia.no/english/studies/courses/2026/spring/ikt460.html",
]

courses = []

for url in urls:

    # Download page
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Course title
    title = soup.find("h1").get_text(strip=True)

    # ECTS credits
    ects = soup.find(string="ECTS Credits:")
    ects_value = ects.find_next().get_text(strip=True)
    
    # Course leader
    leader = soup.find(string="Course Leader:")
    leader_value = leader.find_next().get_text(strip=True)
    
    """ # Teaching language
    language = soup.find(string="Teaching language")
    language_value = language.find_next("p").get_text(strip=True)"""
    
    # Learning outcomes section
    learning_section = soup.find("h2", string="Learning outcomes")
    learning_outcomes = learning_section.find_next("ul").get_text("\n", strip=True)
    
    # Contents section
    contents_section = soup.find("h2", string="Contents")
    contents_text = contents_section.find_next("p").get_text(strip=True)
    
    course_data = {
    "title": title,
    "ects": ects_value,
    "course_leader": leader_value,
    #"language": language_value,
    "learning_outcomes": learning_outcomes,
    "contents": contents_text
}
    courses.append(course_data)

# Save results
df = pd.DataFrame(courses)
df.to_csv("uia_ikt_courses.csv", index=False)

print(df)

In [ ]:
#4. Chunking and indexing RUN FROM HERE
import pandas as pd

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1) Load your scraped data
df = pd.read_csv("uia_ikt_courses.csv")

# 2) Turn each row into a LangChain Document
documents = []

for i, row in df.iterrows():
    title = str(row.get("title", ""))
    ects = str(row.get("ects", ""))
    leader = str(row.get("course_leader", ""))

    # If you also scraped more fields later, add them here
    content = f"""
Title: {row['title']}
ECTS: {row['ects']}
Course leader: {row['course_leader']}

Learning outcomes:
{row.get('learning_outcomes', '')}

Course contents:
{row.get('contents', '')}
""".strip()

    documents.append(
        Document(
            page_content=content,
            metadata={
                "row_id": i,
                "title": title,
                "ects": ects,
                "course_leader": leader,
            },
        )
    )

# 3) Chunk the documents
# For course descriptions, small chunks are usually enough
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Original documents: {len(documents)}")
print(f"Chunks created: {len(chunks)}")

# 4) Create a CPU-only embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# 5) Index chunks in local Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db",
    collection_name="uia_courses",
)

print("Chunking and indexing complete.")